# Train & Evaluate Model - Heart Disease Classification

Notebook này là bước **03** trong pipeline sau EDA và Preprocessing.

Pipeline chuẩn của project:

1. `01_eda.ipynb`: phân tích dữ liệu và rút ra quyết định xử lý.
2. `02_Preprocessing.ipynb`: tạo dữ liệu đã xử lý trong `data/processed/`.
3. `03_Train_Evalute_Model.ipynb`: đọc dữ liệu đã xử lý, huấn luyện và đánh giá mô hình.

Notebook này **không lặp lại preprocessing từ raw data** để tránh lệch pipeline và tránh data leakage. Dữ liệu training/test được đọc trực tiếp từ `data/processed/`.


In [ ]:
# Import thư viện cần thiết cho bước training/evaluation
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Style biểu đồ thống nhất với các notebook trước
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
PAIRED_COLORS = plt.get_cmap("Paired").colors

# Đường dẫn theo project structure
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print(f"Processed data directory: {PROCESSED_DIR}")

## 1. Load Processed Data

Dữ liệu này phải được tạo từ `02_Preprocessing.ipynb`:

- `X_train.csv`, `y_train.csv`: dùng để fit model.
- `X_test.csv`, `y_test.csv`: chỉ dùng để đánh giá cuối cùng.

Nếu các file này chưa tồn tại trên nhánh hiện tại, hãy merge/rebase nhánh đã có preprocessing hoặc chạy lại `02_Preprocessing.ipynb` trước.

In [ ]:
required_files = {
    "X_train": PROCESSED_DIR / "X_train.csv",
    "X_test": PROCESSED_DIR / "X_test.csv",
    "y_train": PROCESSED_DIR / "y_train.csv",
    "y_test": PROCESSED_DIR / "y_test.csv",
}

missing_files = [str(path) for path in required_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing processed data files. Run notebooks/02_Preprocessing.ipynb first "
        "or merge the preprocessing branch before running this notebook. Missing files: "
        + ", ".join(missing_files)
    )

X_train = pd.read_csv(required_files["X_train"])
X_test = pd.read_csv(required_files["X_test"])
y_train = pd.read_csv(required_files["y_train"])["target"]
y_test = pd.read_csv(required_files["y_test"])["target"]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")

display(X_train.head())

## 2. Check Train/Test Target Distribution

Kiểm tra nhanh để đảm bảo tỷ lệ class vẫn khớp với preprocessing stratified split.

In [ ]:
target_distribution = pd.DataFrame({
    "split": ["train", "test"],
    "n_samples": [len(y_train), len(y_test)],
    "disease_count": [int(y_train.sum()), int(y_test.sum())],
    "disease_percent": [round(y_train.mean() * 100, 2), round(y_test.mean() * 100, 2)],
})

display(target_distribution)
target_distribution.to_csv(TABLES_DIR / "train_test_target_distribution.csv", index=False)

## 3. Train Random Forest with GridSearchCV

Random Forest được dùng làm model chính ở bước này. Dữ liệu đã được scale/one-hot encode từ notebook preprocessing, nên ở đây chỉ fit model.

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    class_weight="balanced",
)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best params:", grid_search.best_params_)
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

In [ ]:
# Lưu top-10 kết quả cross-validation để đưa vào báo cáo
cv_results = (
    pd.DataFrame(grid_search.cv_results_)
    [["params", "mean_test_score", "std_test_score", "rank_test_score"]]
    .sort_values("rank_test_score")
    .head(10)
    .reset_index(drop=True)
)

display(cv_results)
cv_results.to_csv(TABLES_DIR / "rf_cv_results_top10.csv", index=False)

## 4. Evaluate on Test Set

Test set chỉ được dùng ở bước này để đánh giá model sau khi đã chọn hyperparameter bằng cross-validation trên train.

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": round(accuracy_score(y_test, y_pred), 4),
    "Precision": round(precision_score(y_test, y_pred), 4),
    "Recall": round(recall_score(y_test, y_pred), 4),
    "F1-Score": round(f1_score(y_test, y_pred), 4),
    "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4),
}

metrics_df = pd.DataFrame(metrics.items(), columns=["Metric", "Value"])
display(metrics_df)
metrics_df.to_csv(TABLES_DIR / "rf_test_metrics.csv", index=False)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

### 4.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Disease", "Disease"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix - Random Forest")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rf_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn} | FP={fp} | FN={fn} | TP={tp}")

### 4.2 ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = metrics["ROC-AUC"]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color=PAIRED_COLORS[5], lw=2, label=f"ROC curve (AUC = {auc_score:.4f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve - Random Forest")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rf_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.3 Feature Importance

Vì `X_train.csv` đã có tên cột sau preprocessing, feature importance có thể dùng trực tiếp các cột này.

In [ ]:
fi_df = (
    pd.DataFrame({"feature": X_train.columns, "importance": best_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(fi_df.head(15))
fi_df.to_csv(TABLES_DIR / "rf_feature_importance.csv", index=False)

top15 = fi_df.head(15)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top15,
    y="feature",
    x="importance",
    hue="feature",
    palette="Blues_r",
    legend=False,
    ax=ax,
)
ax.set_title("Top-15 Feature Importances - Random Forest")
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rf_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.4 Threshold Tuning

Trong bài toán y tế, có thể ưu tiên recall để giảm false negative. Phần này quét threshold để so sánh precision, recall và F1.

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.01)
threshold_rows = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(y_test, y_pred_threshold, zero_division=0),
        "recall": recall_score(y_test, y_pred_threshold, zero_division=0),
        "f1": f1_score(y_test, y_pred_threshold, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
best_threshold = threshold_df.loc[threshold_df["f1"].idxmax()]

display(best_threshold.to_frame().T)
threshold_df.to_csv(TABLES_DIR / "rf_threshold_tuning.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision", color=PAIRED_COLORS[1])
ax.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall", color=PAIRED_COLORS[5])
ax.plot(threshold_df["threshold"], threshold_df["f1"], label="F1-Score", color=PAIRED_COLORS[3], lw=2)
ax.axvline(best_threshold["threshold"], color="gray", linestyle="--", label=f"Best threshold = {best_threshold['threshold']}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 vs. Decision Threshold")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rf_threshold_tuning.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Save Model and Summary

In [ ]:
model_path = MODELS_DIR / "random_forest_model.joblib"
joblib.dump(best_model, model_path)

summary_lines = [
    "# Model Training & Evaluation Summary - Random Forest",
    "",
    "## Input Data",
    "- Training features: `data/processed/X_train.csv`",
    "- Training target: `data/processed/y_train.csv`",
    "- Test features: `data/processed/X_test.csv`",
    "- Test target: `data/processed/y_test.csv`",
    "",
    "## Best Hyperparameters",
]

for key, value in grid_search.best_params_.items():
    summary_lines.append(f"- `{key}`: {value}")

summary_lines += [
    "",
    f"- Best CV ROC-AUC: **{grid_search.best_score_:.4f}**",
    "",
    "## Test-Set Metrics",
    "",
    metrics_df.to_markdown(index=False),
    "",
    "## Threshold Tuning",
    f"- Optimal decision threshold by F1: **{best_threshold['threshold']}**",
    f"- Precision: {best_threshold['precision']:.4f}",
    f"- Recall: {best_threshold['recall']:.4f}",
    f"- F1: {best_threshold['f1']:.4f}",
    "",
    "## Top-5 Features by Importance",
    "",
    fi_df.head(5).to_markdown(index=False),
]

summary_text = "\n".join(summary_lines) + "\n"
summary_path = REPORTS_DIR / "rf_model_summary.md"
summary_path.write_text(summary_text, encoding="utf-8")

print(f"Saved model to: {model_path}")
print(f"Saved model summary to: {summary_path}")